# 🧬 Sequence Alignment with Dynamic Programming

## Learning Objectives

1. Implement Needleman-Wunsch global alignment
2. Implement Smith-Waterman local alignment
3. Understand scoring matrices and gap penalties

---

This is the **most important** DP application in bioinformatics!

In [ ]:
import numpy as np

def needleman_wunsch(seq1, seq2, match=1, mismatch=-1, gap=-2):
    """
    Global sequence alignment.
    Returns: (score, aligned_seq1, aligned_seq2)
    """
    m, n = len(seq1), len(seq2)
    
    # Initialize scoring matrix
    dp = np.zeros((m + 1, n + 1), dtype=int)
    for i in range(m + 1):
        dp[i][0] = i * gap
    for j in range(n + 1):
        dp[0][j] = j * gap
    
    # Fill matrix
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            score = match if seq1[i-1] == seq2[j-1] else mismatch
            dp[i][j] = max(
                dp[i-1][j-1] + score,  # Match/mismatch
                dp[i-1][j] + gap,       # Gap in seq2
                dp[i][j-1] + gap        # Gap in seq1
            )
    
    # Traceback
    align1, align2 = [], []
    i, j = m, n
    while i > 0 or j > 0:
        if i > 0 and j > 0:
            score = match if seq1[i-1] == seq2[j-1] else mismatch
            if dp[i][j] == dp[i-1][j-1] + score:
                align1.append(seq1[i-1])
                align2.append(seq2[j-1])
                i -= 1
                j -= 1
                continue
        if i > 0 and dp[i][j] == dp[i-1][j] + gap:
            align1.append(seq1[i-1])
            align2.append('-')
            i -= 1
        else:
            align1.append('-')
            align2.append(seq2[j-1])
            j -= 1
    
    return dp[m][n], ''.join(reversed(align1)), ''.join(reversed(align2))

seq1 = "ATCGTAC"
seq2 = "ATGTTAT"
score, a1, a2 = needleman_wunsch(seq1, seq2)

print(f"Global Alignment (Needleman-Wunsch)")
print(f"Score: {score}")
print(f"Seq1: {a1}")
print(f"Seq2: {a2}")

In [ ]:
def smith_waterman(seq1, seq2, match=2, mismatch=-1, gap=-1):
    """
    Local sequence alignment.
    Returns: (score, aligned_seq1, aligned_seq2, start_positions)
    """
    m, n = len(seq1), len(seq2)
    dp = np.zeros((m + 1, n + 1), dtype=int)
    
    max_score = 0
    max_pos = (0, 0)
    
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            score = match if seq1[i-1] == seq2[j-1] else mismatch
            dp[i][j] = max(
                0,  # Local: can restart
                dp[i-1][j-1] + score,
                dp[i-1][j] + gap,
                dp[i][j-1] + gap
            )
            if dp[i][j] > max_score:
                max_score = dp[i][j]
                max_pos = (i, j)
    
    # Traceback from max score position
    align1, align2 = [], []
    i, j = max_pos
    while dp[i][j] > 0:
        score = match if seq1[i-1] == seq2[j-1] else mismatch
        if dp[i][j] == dp[i-1][j-1] + score:
            align1.append(seq1[i-1])
            align2.append(seq2[j-1])
            i -= 1
            j -= 1
        elif dp[i][j] == dp[i-1][j] + gap:
            align1.append(seq1[i-1])
            align2.append('-')
            i -= 1
        else:
            align1.append('-')
            align2.append(seq2[j-1])
            j -= 1
    
    return max_score, ''.join(reversed(align1)), ''.join(reversed(align2))

# Find local alignment in longer sequences
seq1 = "AAATCGTACGGG"
seq2 = "CCATGTTATCC"
score, a1, a2 = smith_waterman(seq1, seq2)

print(f"\nLocal Alignment (Smith-Waterman)")
print(f"Score: {score}")
print(f"Seq1: {a1}")
print(f"Seq2: {a2}")

## Global vs Local Alignment

| Aspect | Needleman-Wunsch | Smith-Waterman |
|--------|-----------------|----------------|
| Type | Global | Local |
| Initialization | Gap penalties | Zeros |
| Minimum score | Negative allowed | Zero (restart) |
| Traceback start | Bottom-right | Maximum cell |
| Use case | Similar length sequences | Finding conserved regions |

---

**This completes the Dynamic Programming module!**